# Paper 2 — Phase 0 canonical data audit and analysis freeze

**Purpose.** Build the governed Paper 2 analysis ledgers *before* fitting any clinical model.

This notebook:

1. inventories and hashes the frozen inputs;
2. verifies the required 573 → 519 / 224 / 158 / 66 / 418 / 101 cohort contract;
3. constructs the retained recording ledger;
4. deterministically selects the Goal 1 diagnosis index recording;
5. reconstructs dated ALSFRS-R bulbar assessments and nearest recording-assessment matches;
6. selects the Goal 1 severity index pair;
7. audits Core-Q support/missingness;
8. writes the Paper 2 Q registry;
9. creates the fixed 5-fold × 10-repeat participant split manifest;
10. reports unresolved pre-modeling issues.

**No clinical model is fitted here.** Any failed hard assertion stops the analysis rather than silently changing the cohort.


In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold
from IPython.display import display, Markdown

BASE_SEED = 20260825
OUTER_FOLDS = 5
OUTER_REPEATS = 10

EXPECTED = {
    "source_bamboo_recordings": 573,
    "retained_recordings": 519,
    "retained_participants": 224,
    "als_participants": 158,
    "control_participants": 66,
    "als_recordings": 418,
    "control_recordings": 101,
}

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw from the current directory. "
        "Run this notebook from the repo root or notebooks/."
    )

PROJECT_ROOT = find_project_root()
RAW = PROJECT_ROOT / "data" / "raw"
INTERIM = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
MANIFESTS = PROJECT_ROOT / "data" / "manifests"

for directory in [INTERIM, PROCESSED, MANIFESTS]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW:", RAW)
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)


PROJECT_ROOT: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
RAW: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\data\raw
Python: 3.14.2
pandas: 3.0.5
scikit-learn: 1.9.0


## 1. Resolve authoritative inputs and create an immutable input inventory

For Phase 0, the authoritative inputs are the governed Paper 1 Bamboo ledger, reviewed 519-recording feature release, reviewed segmentation decisions, and feature registry.

Hashes are recorded before any derived table is created.


In [2]:
PATHS = {
    "bamboo_ledger": RAW / "data" / "bamboo_recording_freeze_ledger.csv",
    "feature_release_csv": RAW / "features" / "recording_features.csv",
    "feature_release_parquet": RAW / "features" / "recording_features.parquet",
    "feature_registry": RAW / "features" / "feature_registry.csv",
    "segmentation_decisions": RAW / "segments" / "frozen_segmentation_decisions.csv",
    "segmentation_intervals": RAW / "segments" / "frozen_segmentation_intervals.csv",
    "data_freeze_manifest": RAW / "data" / "data_freeze_manifest.json",
    "feature_release_manifest": RAW / "features" / "release_manifest.csv",
    "segmentation_freeze_manifest": RAW / "segments" / "segmentation_freeze_manifest.json",
}

required = [
    "bamboo_ledger",
    "feature_release_csv",
    "feature_registry",
    "segmentation_decisions",
    "segmentation_intervals",
]
missing = [name for name in required if not PATHS[name].exists()]
assert not missing, f"Missing required Phase 0 inputs: {missing}"

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

inventory_rows = []
for path in sorted(p for p in RAW.rglob("*") if p.is_file()):
    inventory_rows.append({
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

input_inventory = pd.DataFrame(inventory_rows)
input_inventory.to_csv(MANIFESTS / "input_inventory.csv", index=False)

print(f"Inventoried {len(input_inventory)} files.")
display(input_inventory.head(20))


Inventoried 88 files.


,relative_path,bytes,sha256
0,data\raw\.gitkeep,0,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...
1,data\raw\data\bamboo_recording_freeze_ledger.csv,573257,96d68c52b5f36ef3ba1a3d0ee6ca879fce036fb449c5a6...
2,data\raw\data\data_freeze_manifest.json,6827,0f7e56318b7fa080d1d2e2df26f233a51f0b43b13cedc4...
3,data\raw\data\diagnosis_provenance.csv,22052,c39cfe903271fd22741368bbf1ff351a0173208990632b...
4,data\raw\data\freeze_summary.csv,245,f164df9fc6d6e5dfbe0763bee6c6f1f84600973351d654...
5,data\raw\data\frozen_bamboo_recordings.csv,573257,96d68c52b5f36ef3ba1a3d0ee6ca879fce036fb449c5a6...
6,data\raw\data\frozen_exact_bamboo_rest_pairs.csv,74072,2441710e06e7b50285c9ba834ff2cdb42ed38707a8b950...
7,data\raw\data\frozen_rest_recordings.csv,393225,b0144731789d77c4a5765ddf70d5f1f4c9e04f34c04193...
8,data\raw\data\metadata_issue_dispositions.csv,72421,8db8aa945ba2ba4466514f5fe3861c6880ccd8f8115b24...
9,data\raw\data\README.md,400,17a3cac6de36c8d300771f5aa4f6f7f824b49cb15f850c...


## 2. Load governed tables and assert the canonical cohort

These are **hard stops**. We do not coerce the data to match the expected denominators.


In [3]:
bamboo = pd.read_csv(PATHS["bamboo_ledger"], low_memory=False)
features = pd.read_csv(PATHS["feature_release_csv"], low_memory=False)
feature_registry_source = pd.read_csv(PATHS["feature_registry"], low_memory=False)
segmentation = pd.read_csv(PATHS["segmentation_decisions"], low_memory=False)

for frame_name, frame in {
    "bamboo": bamboo,
    "features": features,
    "segmentation": segmentation,
}.items():
    assert "logical_recording_id" in frame.columns, f"{frame_name}: missing logical_recording_id"
    assert not frame["logical_recording_id"].duplicated().any(), f"{frame_name}: duplicated logical_recording_id"

assert len(bamboo) == EXPECTED["source_bamboo_recordings"], (len(bamboo), EXPECTED)
assert len(features) == EXPECTED["retained_recordings"], (len(features), EXPECTED)

eligible_seg = segmentation.loc[
    segmentation["segmentation_analysis_eligible"].astype(bool)
].copy()

assert len(eligible_seg) == EXPECTED["retained_recordings"]
assert set(features["logical_recording_id"]) == set(eligible_seg["logical_recording_id"])
assert set(features["logical_recording_id"]).issubset(set(bamboo["logical_recording_id"]))

retained_meta = bamboo.loc[
    bamboo["logical_recording_id"].isin(features["logical_recording_id"])
].copy()

assert len(retained_meta) == EXPECTED["retained_recordings"]
assert retained_meta["SubjectID"].nunique() == EXPECTED["retained_participants"]

dx_summary = (
    retained_meta.groupby("diagnosis_analysis", dropna=False)
    .agg(
        recordings=("logical_recording_id", "size"),
        participants=("SubjectID", "nunique"),
    )
)

assert int(dx_summary.loc["ALS", "recordings"]) == EXPECTED["als_recordings"]
assert int(dx_summary.loc["CONTROLS", "recordings"]) == EXPECTED["control_recordings"]
assert int(dx_summary.loc["ALS", "participants"]) == EXPECTED["als_participants"]
assert int(dx_summary.loc["CONTROLS", "participants"]) == EXPECTED["control_participants"]

print("HARD COHORT ASSERTIONS: PASS")
display(dx_summary)


HARD COHORT ASSERTIONS: PASS


,recordings,participants
diagnosis_analysis,,
ALS,418,158
CONTROLS,101,66


## 3. Build the retained recording table

Only analysis-relevant metadata are carried forward. Raw date of birth and other unnecessary source fields are not propagated into the Paper 2 analysis table.


In [4]:
meta_cols = [
    "logical_recording_id",
    "SubjectID",
    "diagnosis_analysis",
    "recording_date_analysis",
    "Recording date",
    "Assessment date",
    "ALSFRS total score",
    "ALSFRS bulbar subscore",
    "age_at_recording_years",
    "Sex",
    "media_path",
    "media_sha256",
    "freeze_included",
]

missing_meta_cols = [c for c in meta_cols if c not in retained_meta.columns]
assert not missing_meta_cols, f"Required metadata columns missing: {missing_meta_cols}"

# Avoid duplicating metadata/date columns that are also present in the feature release.
feature_payload = features.drop(
    columns=[
        c for c in ["recording_date_analysis", "Recording date"]
        if c in features.columns
    ]
)

recording_table = (
    retained_meta[meta_cols]
    .merge(
        feature_payload,
        on=["logical_recording_id", "SubjectID"],
        how="inner",
        validate="one_to_one",
    )
)

recording_table["participant_id"] = recording_table["SubjectID"].astype("string").str.strip()
recording_table["diagnosis"] = recording_table["diagnosis_analysis"].astype("string").str.strip()
recording_table["recording_date"] = pd.to_datetime(
    recording_table["recording_date_analysis"], errors="raise"
)

assert len(recording_table) == EXPECTED["retained_recordings"]
assert recording_table["participant_id"].nunique() == EXPECTED["retained_participants"]

recording_table.to_csv(PROCESSED / "recording_table_phase0.csv", index=False)

print("recording_table:", recording_table.shape)
display(recording_table[[
    "logical_recording_id", "participant_id", "diagnosis", "recording_date"
]].head())


recording_table: (519, 61)


,logical_recording_id,participant_id,diagnosis,recording_date
0,C02_1075_1_20260420_240_PSG_BAMBOO,C02,CONTROLS,2026-04-20
1,C02_1075_2_20260421_240_PSG_BAMBOO,C02,CONTROLS,2026-04-21
2,C03_1075_1_20260412_240_PSG_BAMBOO,C03,CONTROLS,2026-04-12
3,C03_1075_2_20260423_240_PSG_BAMBOO,C03,CONTROLS,2026-04-23
4,C04_1075_1_20260419_240_PSG_BAMBOO,C04,CONTROLS,2026-04-19


## 4. Deterministic Goal 1 diagnosis index recording

Rule: first chronological retained Bamboo recording per participant. If recording dates tie, choose ascending `logical_recording_id`. This implements the v1.1 addendum's deterministic tie-break in the absence of a finer creation timestamp in the frozen ledger.


In [5]:
diagnosis_index = (
    recording_table
    .sort_values(["participant_id", "recording_date", "logical_recording_id"])
    .drop_duplicates("participant_id", keep="first")
    .copy()
)

assert len(diagnosis_index) == EXPECTED["retained_participants"]
assert diagnosis_index["participant_id"].is_unique

# Document actual same-date ties before the recording-ID tie-break.
first_dates = recording_table.groupby("participant_id")["recording_date"].transform("min")
earliest_candidates = recording_table.loc[recording_table["recording_date"].eq(first_dates)]
tie_counts = earliest_candidates.groupby("participant_id").size()
diagnosis_ties = tie_counts.loc[tie_counts > 1]

diagnosis_index.to_csv(PROCESSED / "goal1_diagnosis_index.csv", index=False)

print(f"Diagnosis index participants: {len(diagnosis_index)}")
print(f"Participants requiring same-date tie-break: {len(diagnosis_ties)}")
if len(diagnosis_ties):
    display(
        earliest_candidates.loc[
            earliest_candidates["participant_id"].isin(diagnosis_ties.index),
            ["participant_id", "recording_date", "logical_recording_id"]
        ].sort_values(["participant_id", "logical_recording_id"])
    )


Diagnosis index participants: 224
Participants requiring same-date tie-break: 1


,participant_id,recording_date,logical_recording_id
6,C05-1,2026-04-20,C05-1_1075_1_20260420_240_PSG_BAMBOO
7,C05-1,2026-04-20,C05-1_1075_2_20260420_240_PSG_BAMBOO


## 5. Reconstruct the dated ALSFRS-R bulbar assessment ledger and nearest matches

The 573-recording governed Bamboo ledger contains repeated dated clinical assessments. We first deduplicate participant/date/score combinations to form an assessment ledger, then match **each retained ALS recording** to that participant's nearest assessment.

Tie-break for equally close assessments: earlier assessment date.

`severity_pairs.csv` retains the nearest match within 90 days and flags the primary ≤60-day subset, so the 90-day sensitivity can reuse the same frozen matching logic.


In [6]:
assessment_source = bamboo.loc[
    bamboo["diagnosis_analysis"].eq("ALS"),
    ["SubjectID", "Assessment date", "ALSFRS bulbar subscore", "ALSFRS total score"]
].copy()

assessment_source["participant_id"] = assessment_source["SubjectID"].astype("string").str.strip()
assessment_source["assessment_date"] = pd.to_datetime(
    assessment_source["Assessment date"], errors="coerce"
)
assessment_source["bulbar_score"] = pd.to_numeric(
    assessment_source["ALSFRS bulbar subscore"], errors="coerce"
)
assessment_source["alsfrs_total"] = pd.to_numeric(
    assessment_source["ALSFRS total score"], errors="coerce"
)

assessment_source = assessment_source.dropna(
    subset=["participant_id", "assessment_date", "bulbar_score"]
)

# A participant/date cannot carry conflicting bulbar scores.
conflicts = (
    assessment_source.groupby(["participant_id", "assessment_date"])["bulbar_score"]
    .nunique()
)
assert not (conflicts > 1).any(), (
    "Conflicting ALSFRS-R bulbar values found for the same participant/date."
)

assessment_table = (
    assessment_source[
        ["participant_id", "assessment_date", "bulbar_score", "alsfrs_total"]
    ]
    .drop_duplicates()
    .sort_values(["participant_id", "assessment_date"])
    .reset_index(drop=True)
)

als_recordings = recording_table.loc[
    recording_table["diagnosis"].eq("ALS"),
    ["participant_id", "logical_recording_id", "recording_date"]
].copy()

candidate_pairs = als_recordings.merge(
    assessment_table, on="participant_id", how="left", validate="many_to_many"
)
candidate_pairs["delta_days"] = (
    candidate_pairs["assessment_date"] - candidate_pairs["recording_date"]
).dt.days
candidate_pairs["abs_delta_days"] = candidate_pairs["delta_days"].abs()

# Keep candidates in the sensitivity horizon, then choose nearest per recording.
nearest_90 = (
    candidate_pairs.loc[candidate_pairs["abs_delta_days"].le(90)]
    .sort_values([
        "participant_id",
        "logical_recording_id",
        "abs_delta_days",
        "assessment_date",  # earlier date wins an equal-distance tie
    ])
    .drop_duplicates("logical_recording_id", keep="first")
    .copy()
)

nearest_90["within_60_days"] = nearest_90["abs_delta_days"].le(60)
nearest_90["within_90_days"] = True

severity_pairs = nearest_90.sort_values(
    ["participant_id", "recording_date", "logical_recording_id"]
).reset_index(drop=True)

severity_pairs.to_csv(PROCESSED / "severity_pairs.csv", index=False)

primary_pairs = severity_pairs.loc[severity_pairs["within_60_days"]].copy()

severity_index = (
    primary_pairs
    .sort_values(["participant_id", "recording_date", "logical_recording_id"])
    .drop_duplicates("participant_id", keep="first")
    .copy()
)

severity_index.to_csv(PROCESSED / "goal1_severity_index.csv", index=False)

print("Unique dated ALS assessments:", len(assessment_table))
print("ALS participants represented in assessment ledger:", assessment_table["participant_id"].nunique())
print("Retained recording-assessment pairs <=60 d:", len(primary_pairs))
print("ALS participants matched <=60 d:", primary_pairs["participant_id"].nunique())
print("Goal 1 severity index N:", len(severity_index))
print("ALS participants matched <=90 d:", severity_pairs["participant_id"].nunique())

display(severity_index[[
    "participant_id", "logical_recording_id", "recording_date",
    "assessment_date", "abs_delta_days", "bulbar_score"
]].head())


Unique dated ALS assessments: 289
ALS participants represented in assessment ledger: 153
Retained recording-assessment pairs <=60 d: 398
ALS participants matched <=60 d: 145
Goal 1 severity index N: 145
ALS participants matched <=90 d: 145


,participant_id,logical_recording_id,recording_date,assessment_date,abs_delta_days,bulbar_score
0,CAPT0000001,CAPT0000001_272_1_20220518_240_PSG_BAMBOO,2022-05-18,2022-05-18,0.0,9.0
1,CAPT0000002,CAPT0000002_272_2_20221114_240_PSG_BAMBOO,2022-11-14,2022-11-14,0.0,12.0
2,CAPT0000004,CAPT0000004_272_2_20221208_240_PSG_BAMBOO,2022-12-08,2022-12-08,0.0,12.0
3,CAPT0000005,CAPT0000005_272_1_20220817_240_PSG_BAMBOO,2022-08-17,2022-08-17,0.0,8.0
7,CAPT0000010,CAPT0000010_272_1_20230330_240_PSG_BAMBOO,2023-03-30,2023-03-30,0.0,9.0


## 6. Participant table and demographic completeness audit

Diagnosis itself is complete. Demographic benchmarks require a separate completeness check because missing demographic metadata must not silently change the primary Core-Q cohort.


In [7]:
participant_table = (
    diagnosis_index[[
        "participant_id",
        "diagnosis",
        "logical_recording_id",
        "recording_date",
        "age_at_recording_years",
        "Sex",
    ]]
    .rename(columns={
        "logical_recording_id": "diagnosis_index_recording_id",
        "recording_date": "diagnosis_index_recording_date",
        "age_at_recording_years": "age_at_index_recording_years",
        "Sex": "sex",
    })
    .copy()
)

n_recordings = (
    recording_table.groupby("participant_id")
    .size()
    .rename("n_retained_recordings")
    .reset_index()
)
participant_table = participant_table.merge(
    n_recordings, on="participant_id", how="left", validate="one_to_one"
)

participant_table["has_primary_bulbar_pair"] = participant_table["participant_id"].isin(
    set(severity_index["participant_id"])
)

participant_table.to_csv(PROCESSED / "participant_table.csv", index=False)

demo_audit = (
    participant_table.groupby("diagnosis")
    .agg(
        participants=("participant_id", "size"),
        age_available=("age_at_index_recording_years", lambda x: int(x.notna().sum())),
        sex_available=("sex", lambda x: int(x.notna().sum())),
    )
)
demo_audit["age_missing"] = demo_audit["participants"] - demo_audit["age_available"]
demo_audit["sex_missing"] = demo_audit["participants"] - demo_audit["sex_available"]

display(demo_audit)

if demo_audit.loc["CONTROLS", "age_missing"] > 0:
    display(Markdown(
        "**PRE-MODELING ISSUE — age benchmark:** control age is incomplete. "
        "Do not let age missingness become an implicit diagnosis predictor. "
        "Before the Age-only / Age+Core-Q benchmark is finalized, either recover a verified age/DOB source "
        "or freeze an explicit complete-case benchmark rule."
    ))


,participants,age_available,sex_available,age_missing,sex_missing
diagnosis,,,,,
ALS,158,158,158,0,0
CONTROLS,66,41,42,25,24


**PRE-MODELING ISSUE — age benchmark:** control age is incomplete. Do not let age missingness become an implicit diagnosis predictor. Before the Age-only / Age+Core-Q benchmark is finalized, either recover a verified age/DOB source or freeze an explicit complete-case benchmark rule.

## 7. Freeze Core-Q / Extended-Q registry and audit support

The Paper 2 Methods Reference defines **Core-Q = 12 numeric features + 3 QADD support indicators**.

The transforms below reproduce the frozen Paper 1 analysis transforms (`none`, `log1p`, `asinh`) and are not selected from Paper 2 outcomes.

QCHAN values in the 519-recording release remain useful for cohort auditing, but **must not be used directly as final predictive inputs** because Paper 2 requires rebuilding the empirical QCHAN reference using outer-training participants only.


In [8]:
CORE_Q = [
    # QADD
    "qadd_pause_ac_level_dbfs_median",
    "qadd_pause_level_iqr_db",
    "qadd_speech_pause_level_contrast_db",
    # QGAIN
    "qgain_typical_speech_level_dbfs",
    "qgain_within_segment_iqr_db",
    "qgain_between_segment_mad_db",
    "qgain_abs_drift_db_per_min",
    # QREV
    "qrev_srmr_norm",
    # QCHAN
    "qchan_ltas_distance_db",
    "qchan_rolloff95_deficit_hz",
    "qchan_highband_ratio_deficit",
    "qchan_tilt_steepening_db_per_oct",
]

EXTENDED_Q = [
    "qadd_pause_spectral_flatness",
    "qadd_mains_hum_comb_score_db",
    "qrev_tail_excess_100ms_db",
    "qrev_tail_persistence_median_sec",
    "qrev_downward_decay_rate_db_per_sec",
]

TRANSFORMS = {
    "qadd_pause_ac_level_dbfs_median": "none",
    "qadd_pause_level_iqr_db": "log1p",
    "qadd_speech_pause_level_contrast_db": "none",
    "qadd_pause_spectral_flatness": "none",
    "qadd_mains_hum_comb_score_db": "asinh",
    "qgain_typical_speech_level_dbfs": "none",
    "qgain_within_segment_iqr_db": "log1p",
    "qgain_between_segment_mad_db": "log1p",
    "qgain_abs_drift_db_per_min": "log1p",
    "qrev_tail_excess_100ms_db": "asinh",
    "qrev_tail_persistence_median_sec": "log1p",
    "qrev_downward_decay_rate_db_per_sec": "log1p",
    "qrev_srmr_norm": "log1p",
    "qchan_ltas_distance_db": "log1p",
    "qchan_rolloff95_deficit_hz": "log1p",
    "qchan_highband_ratio_deficit": "log1p",
    "qchan_tilt_steepening_db_per_oct": "log1p",
}

# Validate every prespecified Paper 2 Q feature exists.
missing_q = [c for c in CORE_Q + EXTENDED_Q if c not in recording_table.columns]
assert not missing_q, f"Missing prespecified Q features: {missing_q}"

# Explicit binary QADD support indicators used by the predictive pipeline.
QADD_SUPPORT_INDICATORS = {}
for feature in CORE_Q[:3]:
    indicator = f"{feature}_supported"
    recording_table[indicator] = recording_table[feature].notna().astype("int8")
    QADD_SUPPORT_INDICATORS[feature] = indicator

support_audit = pd.DataFrame([
    {
        "feature": feature,
        "n_available": int(recording_table[feature].notna().sum()),
        "n_missing": int(recording_table[feature].isna().sum()),
        "fraction_available": float(recording_table[feature].notna().mean()),
        "n_zero": int((recording_table[feature] == 0).sum()),
    }
    for feature in CORE_Q
])

source_registry = feature_registry_source.set_index("feature")

registry_rows = []
for feature in CORE_Q + EXTENDED_Q:
    src = source_registry.loc[feature]
    registry_rows.append({
        "feature": feature,
        "family": src["family_code"],
        "measurement_version": src["measurement_version"],
        "unit": src["unit"],
        "paper2_role": "core" if feature in CORE_Q else "extended",
        "transform": TRANSFORMS[feature],
        "status_field": src["status_field"],
        "support_indicator": QADD_SUPPORT_INDICATORS.get(feature),
        "missing_rule": (
            "outer-training median + binary support indicator"
            if feature in QADD_SUPPORT_INDICATORS
            else "no primary missingness expected; fail/audit if encountered"
        ),
        "zero_semantics": (
            "one-sided registered zero; not clean/unaltered"
            if feature.startswith("qchan_") and feature != "qchan_ltas_distance_db"
            else "continuous registered measurement"
        ),
        "requires_outer_training_reference": feature.startswith("qchan_"),
    })

q_registry = pd.DataFrame(registry_rows)
q_registry.to_csv(MANIFESTS / "q_registry.csv", index=False)

# Re-save recording table with generated support indicators.
recording_table.to_csv(PROCESSED / "recording_table_phase0.csv", index=False)

display(support_audit)
display(q_registry)


,feature,n_available,n_missing,fraction_available,n_zero
0,qadd_pause_ac_level_dbfs_median,462,57,0.890173,0
1,qadd_pause_level_iqr_db,440,79,0.847784,0
2,qadd_speech_pause_level_contrast_db,462,57,0.890173,0
3,qgain_typical_speech_level_dbfs,519,0,1.000000,0
4,qgain_within_segment_iqr_db,519,0,1.000000,0
5,qgain_between_segment_mad_db,519,0,1.000000,0
6,qgain_abs_drift_db_per_min,519,0,1.000000,0
7,qrev_srmr_norm,519,0,1.000000,0
8,qchan_ltas_distance_db,519,0,1.000000,0
9,qchan_rolloff95_deficit_hz,519,0,1.000000,264


,feature,family,measurement_version,unit,paper2_role,transform,status_field,support_indicator,missing_rule,zero_semantics,requires_outer_training_reference
0,qadd_pause_ac_level_dbfs_median,QADD,qadd-v4.2.0,dBFS,core,none,qadd_pause_ac_level_dbfs_median_status,qadd_pause_ac_level_dbfs_median_supported,outer-training median + binary support indicator,continuous registered measurement,False
1,qadd_pause_level_iqr_db,QADD,qadd-v4.2.0,dB,core,log1p,qadd_pause_level_iqr_db_status,qadd_pause_level_iqr_db_supported,outer-training median + binary support indicator,continuous registered measurement,False
2,qadd_speech_pause_level_contrast_db,QADD,qadd-v4.2.0,dB,core,none,qadd_speech_pause_level_contrast_db_status,qadd_speech_pause_level_contrast_db_supported,outer-training median + binary support indicator,continuous registered measurement,False
3,qgain_typical_speech_level_dbfs,QGAIN,qgain-v4.1.0,dBFS,core,none,qgain_typical_speech_level_dbfs_status,NaN,no primary missingness expected; fail/audit if...,continuous registered measurement,False
4,qgain_within_segment_iqr_db,QGAIN,qgain-v4.1.0,dB,core,log1p,qgain_within_segment_iqr_db_status,NaN,no primary missingness expected; fail/audit if...,continuous registered measurement,False
5,qgain_between_segment_mad_db,QGAIN,qgain-v4.1.0,dB,core,log1p,qgain_between_segment_mad_db_status,NaN,no primary missingness expected; fail/audit if...,continuous registered measurement,False
6,qgain_abs_drift_db_per_min,QGAIN,qgain-v4.1.0,dB/min,core,log1p,qgain_abs_drift_db_per_min_status,NaN,no primary missingness expected; fail/audit if...,continuous registered measurement,False
7,qrev_srmr_norm,QREV,qrev-v4.0.0,ratio,core,log1p,qrev_srmr_norm_status,NaN,no primary missingness expected; fail/audit if...,continuous registered measurement,False
8,qchan_ltas_distance_db,QCHAN,qchan-v4.0.0,dB RMS,core,log1p,qchan_ltas_distance_db_status,NaN,no primary missingness expected; fail/audit if...,continuous registered measurement,True
9,qchan_rolloff95_deficit_hz,QCHAN,qchan-v4.0.0,Hz,core,log1p,qchan_rolloff95_deficit_hz_status,NaN,no primary missingness expected; fail/audit if...,one-sided registered zero; not clean/unaltered,True


## 8. Create the fixed participant-level outer split manifest

The split is created **once at participant level**. Every recording later inherits its participant's assignment.

Implementation policy: repeat `r` uses deterministic seed `20260825 + r`, with 5-fold stratified participant splitting. This makes every repeat seed explicit in the manifest.


In [9]:
participants_for_split = (
    participant_table[["participant_id", "diagnosis"]]
    .sort_values("participant_id")
    .reset_index(drop=True)
)

y = participants_for_split["diagnosis"].map({"CONTROLS": 0, "ALS": 1})
assert y.notna().all()

split_rows = []

for repeat in range(OUTER_REPEATS):
    repeat_seed = BASE_SEED + repeat
    splitter = StratifiedKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=repeat_seed,
    )

    for fold, (_, test_idx) in enumerate(
        splitter.split(participants_for_split["participant_id"], y),
        start=1,
    ):
        test = participants_for_split.iloc[test_idx]
        for row in test.itertuples(index=False):
            split_rows.append({
                "participant_id": row.participant_id,
                "diagnosis": row.diagnosis,
                "repeat": repeat + 1,
                "outer_fold": fold,
                "repeat_seed": repeat_seed,
                "base_seed": BASE_SEED,
            })

split_manifest = pd.DataFrame(split_rows)

# Each participant must appear exactly once per repeat.
counts = split_manifest.groupby(["participant_id", "repeat"]).size()
assert counts.eq(1).all()
assert len(split_manifest) == EXPECTED["retained_participants"] * OUTER_REPEATS

split_manifest.to_csv(MANIFESTS / "split_manifest.csv", index=False)

split_audit = (
    split_manifest.groupby(["repeat", "outer_fold", "diagnosis"])
    .size()
    .unstack("diagnosis", fill_value=0)
    .reset_index()
)

print("split_manifest rows:", len(split_manifest))
display(split_audit)


split_manifest rows: 2240


diagnosis,repeat,outer_fold,ALS,CONTROLS
0,1,1,31,14
1,1,2,32,13
2,1,3,32,13
3,1,4,32,13
4,1,5,31,13
5,2,1,31,14
6,2,2,32,13
7,2,3,32,13
8,2,4,32,13
9,2,5,31,13


## 9. Local availability check for future fold-safe QCHAN reconstruction

The frozen Paper 2 specification requires the QCHAN empirical reference to be constructed from the current outer-training participants.

The uploaded release contains final reference-relative QCHAN values, not the upstream reference-independent LTAS/rolloff/high-band/tilt primitives. Therefore Goal 1's QCHAN branch requires either:

- access to the original waveforms plus the validated QCHAN extractor, or
- a frozen reference-independent per-recording QCHAN primitive/cache generated by that extractor.

This cell only checks whether the original `media_path` entries resolve on the current computer. It does not alter the cohort.


In [10]:
media_paths = recording_table["media_path"].dropna().astype(str)
media_exists = media_paths.map(lambda p: Path(p).exists())

qchan_access_audit = {
    "retained_recordings": len(recording_table),
    "media_paths_recorded": int(len(media_paths)),
    "media_paths_existing_on_this_machine": int(media_exists.sum()),
    "all_media_paths_resolve": bool(media_exists.all()) if len(media_exists) else False,
}

print(json.dumps(qchan_access_audit, indent=2))

if not qchan_access_audit["all_media_paths_resolve"]:
    display(Markdown(
        "**QCHAN IMPLEMENTATION GATE:** not all original waveform paths resolve from this environment. "
        "This does not invalidate Phase 0, but final Goal 1 QCHAN/Core-Q modeling must wait until "
        "fold-safe QCHAN reconstruction is operational."
    ))


{
  "retained_recordings": 519,
  "media_paths_recorded": 519,
  "media_paths_existing_on_this_machine": 519,
  "all_media_paths_resolve": true
}


## 10. Freeze report

This is the point where we decide whether Phase 0 is scientifically ready for Goal 1 implementation. A canonical-cohort PASS does **not** automatically imply that every benchmark/preprocessing branch is ready.


In [11]:
status_rows = [
    {
        "gate": "Canonical 573 -> 519 recording flow",
        "status": "PASS",
        "detail": f"{len(bamboo)} source; {len(recording_table)} retained",
    },
    {
        "gate": "Retained participant denominator",
        "status": "PASS",
        "detail": (
            f"{recording_table['participant_id'].nunique()} participants; "
            f"{EXPECTED['als_participants']} ALS / {EXPECTED['control_participants']} controls"
        ),
    },
    {
        "gate": "Goal 1 diagnosis index",
        "status": "PASS",
        "detail": f"{len(diagnosis_index)} deterministic participant rows",
    },
    {
        "gate": "Goal 1 severity <=60 d",
        "status": "PASS",
        "detail": f"{len(severity_index)} ALS participants",
    },
    {
        "gate": "Age benchmark metadata",
        "status": (
            "PASS"
            if participant_table["age_at_index_recording_years"].notna().all()
            else "REQUIRES DECISION/RECOVERY"
        ),
        "detail": (
            f"{participant_table['age_at_index_recording_years'].notna().sum()}/"
            f"{len(participant_table)} participants have age"
        ),
    },
    {
        "gate": "Core-Q registry/support",
        "status": "PASS",
        "detail": "12 numeric Core-Q features + 3 QADD support indicators defined",
    },
    {
        "gate": "Fold-safe QCHAN reconstruction",
        "status": "READY" if qchan_access_audit["all_media_paths_resolve"] else "BLOCKED",
        "detail": (
            "All original waveform paths resolve"
            if qchan_access_audit["all_media_paths_resolve"]
            else "Need original waveforms/path repair or frozen QCHAN primitives"
        ),
    },
    {
        "gate": "Master outer split manifest",
        "status": "PASS",
        "detail": f"{OUTER_FOLDS} folds x {OUTER_REPEATS} repeats; base seed {BASE_SEED}",
    },
]

freeze_report = pd.DataFrame(status_rows)
freeze_report.to_csv(MANIFESTS / "phase0_freeze_report.csv", index=False)

run_metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "base_seed": BASE_SEED,
    "outer_folds": OUTER_FOLDS,
    "outer_repeats": OUTER_REPEATS,
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}
(MANIFESTS / "phase0_run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2), encoding="utf-8"
)

display(freeze_report)

print("\nDerived local artifacts:")
for path in [
    PROCESSED / "recording_table_phase0.csv",
    PROCESSED / "participant_table.csv",
    PROCESSED / "goal1_diagnosis_index.csv",
    PROCESSED / "severity_pairs.csv",
    PROCESSED / "goal1_severity_index.csv",
    MANIFESTS / "input_inventory.csv",
    MANIFESTS / "q_registry.csv",
    MANIFESTS / "split_manifest.csv",
    MANIFESTS / "phase0_freeze_report.csv",
    MANIFESTS / "phase0_run_metadata.json",
]:
    print(" -", path.relative_to(PROJECT_ROOT))


,gate,status,detail
0,Canonical 573 -> 519 recording flow,PASS,573 source; 519 retained
1,Retained participant denominator,PASS,224 participants; 158 ALS / 66 controls
2,Goal 1 diagnosis index,PASS,224 deterministic participant rows
3,Goal 1 severity <=60 d,PASS,145 ALS participants
4,Age benchmark metadata,REQUIRES DECISION/RECOVERY,199/224 participants have age
5,Core-Q registry/support,PASS,12 numeric Core-Q features + 3 QADD support in...
6,Fold-safe QCHAN reconstruction,READY,All original waveform paths resolve
7,Master outer split manifest,PASS,5 folds x 10 repeats; base seed 20260825



Derived local artifacts:
 - data\processed\recording_table_phase0.csv
 - data\processed\participant_table.csv
 - data\processed\goal1_diagnosis_index.csv
 - data\processed\severity_pairs.csv
 - data\processed\goal1_severity_index.csv
 - data\manifests\input_inventory.csv
 - data\manifests\q_registry.csv
 - data\manifests\split_manifest.csv
 - data\manifests\phase0_freeze_report.csv
 - data\manifests\phase0_run_metadata.json
